# 00 - Dataset showcase

A tour of what **3DMPE** can do, loading ground-truth shapes from several
datasets and running the reconstruction pipeline end to end.

This is the modern, self-contained replacement for the old `main.ipynb`. It:

1. loads point clouds from **demo primitives, ModelNet10, ShapeNet (LMNet
   subset) and Pix3D**,
2. optionally previews the source mesh,
3. reconstructs the 3D cloud from simulated 2D views,
4. inspects visibility, the pairwise-distance distribution, a random-cloud
   sanity check, and different alignment strategies.

Everything defaults to a download-free **demo** shape, so the notebook runs as
is. Point the dataset selectors at your local ShapeNet/Pix3D trees (or set the
`SHAPENET_DIR` / `PIX3D_DIR` environment variables) to use the real data.

In [1]:
import os

import numpy as np
import plotly.express as px

from mpe3d import datasets, reconstruct
from mpe3d.alignment import (apply_transformation, four_point_sample_transform,
                             kabsch_transform)
from mpe3d.metrics import chamfer_distance, earth_movers_distance, roa
from mpe3d.visualization import (plot_point_clouds, plot_cost_history,
                                 plot_points_per_perspective)

rng = np.random.default_rng(0)

## 1. Datasets you can load

Every dataset is referenced by a colon-separated selector. The catalogue below
lists a representative example from each supported source; ShapeNet examples are
drawn from the built-in **LMNet** benchmark registry.

In [2]:
SHAPENET_DIR = os.environ.get("SHAPENET_DIR")  # path to ShapeNetCore.v2
PIX3D_DIR = os.environ.get("PIX3D_DIR")         # path to the Pix3D root

catalogue = {
    "demo:torus": None,
    "demo:sphere": None,
    "ModelNet10:chair:0001": None,
    # First LMNet chair / airplane (needs a local ShapeNetCore.v2):
    datasets.lmnet_datasets(categories=["chair"], per_category=1)[0]: SHAPENET_DIR,
    datasets.lmnet_datasets(categories=["airplane"], per_category=1)[0]: SHAPENET_DIR,
    "Pix3D:chair:0132": PIX3D_DIR,
}

for selector, datadir in catalogue.items():
    print(selector)

print("\nLMNet categories:", list(datasets.LMNET_MODELS))
print("Total LMNet models:", sum(len(v) for v in datasets.LMNET_MODELS.values()))

demo:torus
demo:sphere
ModelNet10:chair:0001
ShapeNet:chair:bf91d0169eae3bfdd810b14a81e12eca
ShapeNet:airplane:103c9e43cdf6501c62b600da24e0965
Pix3D:chair:0132

LMNet categories: ['airplane', 'bench', 'car', 'chair', 'lamp', 'rifle', 'table', 'sofa']
Total LMNet models: 36


## 2. Pick an active dataset

We default to a demo torus. Switch `ACTIVE` to any selector above; if the data
is not available locally we fall back to the demo shape so the notebook keeps
running.

In [3]:
ACTIVE = "demo:torus"
DATADIR = catalogue.get(ACTIVE)
N_POINTS = 400

try:
    points = datasets.get_dataset_points(ACTIVE, datadir=DATADIR,
                                         n_points=N_POINTS, normalize=True)
except (FileNotFoundError, AssertionError, ValueError) as err:
    print(f"Could not load {ACTIVE!r} ({err}); falling back to demo:torus.")
    ACTIVE, DATADIR = "demo:torus", None
    points = datasets.get_dataset_points(ACTIVE, n_points=N_POINTS, normalize=True)

print("active dataset:", ACTIVE)
print("point cloud shape:", points.shape)
plot_point_clouds([points], names=[ACTIVE], colors=["fancy"])

active dataset: demo:torus
point cloud shape: (400, 3)


## 3. Preview the source mesh (optional)

Mesh-backed datasets can be previewed before sampling. `trimesh` opens an
interactive viewer with `mesh.show()`; here we just report the mesh stats to keep
the notebook headless-friendly.

In [4]:
try:
    mesh = datasets.get_dataset_mesh(ACTIVE, datadir=DATADIR)
    print(f"{ACTIVE}: {len(mesh.vertices)} vertices, {len(mesh.faces)} faces")
    # mesh.show()  # uncomment for an interactive 3D mesh viewer
except Exception as err:  # noqa: BLE001 - showcase only
    print("No mesh preview available:", err)

demo:torus: 1024 vertices, 2048 faces


## 4. Reconstruct

One call runs the whole pipeline: simulate views, drop points by visibility,
build per-view distance matrices, optimize the MPSE embedding, align it to the
ground truth, and score it (Chamfer / EMD / ROA), alongside the MDS baseline.

In [5]:
result = reconstruct(
    points,
    n_perspectives=5,
    points_in_at_least=3,
    variable_projection=True,
    max_iter=200,
    rng=rng,
)

import json
print(json.dumps(result.summary(), indent=2, default=float))

{
  "chamfer": 20.115306345357958,
  "emd": 16.34763536811557,
  "roa": 0.09420920825869,
  "final_cost": 0.2695725147887733,
  "baseline_chamfer": 12.437614134648395,
  "baseline_emd": 8.944819093449285,
  "baseline_roa": 0.1500804163539984
}


In [6]:
plot_point_clouds(
    [result.ground_truth, result.aligned_embedding],
    names=["ground truth", "reconstruction"],
    colors=["green", "red"],
)

In [7]:
plot_cost_history(result.cost_history)

## 5. How visible is each point?

The pipeline records, for each `k`, how many points are seen in at least `k`
views. 3DMPE reconstructs well as long as most points are visible from **3+**
viewpoints (paper, Figures 7 and 16).

In [8]:
plot_points_per_perspective(result.points_per_perspective)

## 6. Pairwise-distance distribution

A quick sanity check that the reconstruction preserves the geometry: the
distribution of pairwise distances in the reconstruction should track the
ground truth.

In [9]:
from scipy.spatial.distance import pdist

gt_d = pdist(result.ground_truth)
rec_d = pdist(result.aligned_embedding)

px.histogram(
    {"distance": np.concatenate([gt_d, rec_d]),
     "cloud": ["ground truth"] * len(gt_d) + ["reconstruction"] * len(rec_d)},
    x="distance", color="cloud", barmode="overlay", nbins=60,
    title="Pairwise distance distribution",
)

## 7. Random-cloud sanity check

For reference, how bad are the metrics for a *random* cloud spanning the same
range? This is the "no information" floor the reconstruction must beat.

In [10]:
random_cloud = rng.uniform(points.min(), points.max(), size=points.shape)
print("random   Chamfer:", round(chamfer_distance(random_cloud, points), 3),
      "EMD:", round(earth_movers_distance(random_cloud, points), 3))
print("3DMPE    Chamfer:", round(result.chamfer, 3),
      "EMD:", round(result.emd, 3))

random   Chamfer: 17.46 EMD: 13.75
3DMPE    Chamfer: 20.115 EMD: 16.348


## 8. Alignment strategies

Chamfer/EMD ignore rigid pose, so the reconstruction must first be aligned to
the ground truth. We compare the two strategies shipped with 3DMPE:

* **4-point RANSAC** (paper Eq. 6-7) - random 4-point correspondences,
* **Kabsch / SVD** (paper Eq. 13-16) - the closed-form solution used by ROA.

In [11]:
emb = result.embedding

t_4pt, _ = four_point_sample_transform(emb, points, rng=rng)
aligned_4pt = apply_transformation(emb, t_4pt)

t_kab = kabsch_transform(emb, points)
aligned_kab = apply_transformation(emb, t_kab)

print("4-point RANSAC : Chamfer",
      round(chamfer_distance(aligned_4pt, points), 3),
      "| ROA", round(roa(emb, points), 5))
print("Kabsch / SVD   : Chamfer",
      round(chamfer_distance(aligned_kab, points), 3),
      "| ROA", round(roa(emb, points), 5))

plot_point_clouds([points, aligned_kab],
                  names=["ground truth", "Kabsch-aligned"],
                  colors=["green", "red"])

4-point RANSAC : Chamfer 18.378 | ROA 0.09421
Kabsch / SVD   : Chamfer 22.743 | ROA 0.09421


## Where next?

* `01-pipeline-walkthrough.ipynb` - the same pipeline broken down stage by stage.
* `02-noise-experiments.ipynb` - robustness to distance and correspondence noise.
* `03-lmnet-benchmark.ipynb` - the LMNet/ShapeNet parameter sweeps from the paper.